# Preprocess the dataset

We removed irrelevant features as well as features that could potentially introduce future information leakage. Unrealistic values were replaced with NaN for subsequent processing. For example, records with LotSizeSquareFeet = 0 were considered invalid and treated as missing values.

Some features contained a large number of missing values, and directly removing all missing records at the beginning would have resulted in a substantial loss of useful information. Categorical features were converted using one-hot encoding. For high-cardinality categorical features, only the top 10 most frequent categories were retained, while all remaining categories were grouped into an "Other" category before encoding to avoid generating an excessive number of features.

Finally, the remaining records containing missing values were removed. Missing value imputation was not applied during preprocessing because fitting an imputation strategy on the entire dataset before the train–test split could introduce data leakage. Therefore, removing the remaining missing values was adopted as a safer preprocessing strategy.

In [23]:
import pandas as pd
import glob
import os
import numpy as np
path = r"C:\Users\23035\OneDrive\Desktop\Internship\dataset"
files = glob.glob(os.path.join(path, "*.csv"))
dfs = []
for f in files:
    data_month=pd.read_csv(f)
    date_ym=pd.to_datetime(os.path.basename(f).split('.')[0][-6:],format="%Y%m")
    data_month['date_ym']=date_ym
    dfs.append(data_month)
dataset_total_last6 = pd.concat(dfs)

C:\Users\23035\AppData\Local\Temp\ipykernel_22992\3523058081.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  data_month=pd.read_csv(f)
C:\Users\23035\AppData\Local\Temp\ipykernel_22992\3523058081.py:9: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  data_month=pd.read_csv(f)


### 1. Handle missing values and invalid values




In [24]:
# drop the feature which miss more than 10 % values
dataset_total_last6_drop= dataset_total_last6.loc[:,dataset_total_last6.isna().mean() < 0.1]
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 281823 entries, 0 to 23259
Data columns (total 44 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   BuyerAgentAOR             275696 non-null  object        
 1   ListAgentAOR              281707 non-null  object        
 2   OriginalListPrice         280995 non-null  float64       
 3   ListingKey                281823 non-null  int64         
 4   ListAgentEmail            281023 non-null  object        
 5   CloseDate                 281823 non-null  object        
 6   ClosePrice                281820 non-null  float64       
 7   ListAgentFirstName        280536 non-null  object        
 8   ListAgentLastName         281804 non-null  object        
 9   Latitude                  281769 non-null  float64       
 10  Longitude                 281772 non-null  float64       
 11  UnparsedAddress           281404 non-null  object        
 12  Property

In [25]:
# drop future features and unrelated features
# ClosePrice (label)
# We should use the property's features 
# It should be worked on the property whether it is listed or not
dataset_total_last6_drop = dataset_total_last6_drop.drop(columns=[
    # future features (leakage): like the feature related with list, buyer
    "ListAgentAOR",
    "BuyerAgentAOR",# leakage
    "OriginalListPrice",# leakage
    "CloseDate",# future
    "PurchaseContractDate",# future
    "ContractStatusChangeDate",# future
    "DaysOnMarket",# future
    "ListPrice",
    "ListingContractDate",

    # unrelated features and the feature related with List property
    "ListAgentEmail",
    "ListingKey",
    "ListAgentFirstName",
    "ListAgentLastName",
    "BuyerAgentMlsId",
    "ListAgentEmail",
    "ListingKeyNumeric",
    "UnparsedAddress",
    "ListOfficeName",
    "BuyerOfficeName",
    "ListAgentFullName",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "ListingKeyNumeric",
    "MlsStatus",# all Closed - no info
    "BuyerOfficeAOR",
    "ListingId"
])


In [26]:
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 281823 entries, 0 to 23259
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ClosePrice             281820 non-null  float64       
 1   Latitude               281769 non-null  float64       
 2   Longitude              281772 non-null  float64       
 3   PropertyType           281823 non-null  object        
 4   LivingArea             262492 non-null  float64       
 5   CountyOrParish         281823 non-null  object        
 6   ParkingTotal           273866 non-null  float64       
 7   PropertySubType        261493 non-null  object        
 8   LotSizeAcres           257947 non-null  float64       
 9   YearBuilt              270314 non-null  float64       
 10  StreetNumberNumeric    281186 non-null  float64       
 11  BathroomsTotalInteger  269412 non-null  float64       
 12  City                   281740 non-null  object    

In [27]:
# replace unreasonable value
dataset_total_last6_drop["ClosePrice"] = dataset_total_last6_drop["ClosePrice"].replace(0, np.nan)
dataset_total_last6_drop["LotSizeSquareFeet"] = dataset_total_last6_drop["LotSizeSquareFeet"].replace(0, np.nan)
dataset_total_last6_drop["LotSizeAcres"] = dataset_total_last6_drop["LotSizeAcres"].replace(0, np.nan)
dataset_total_last6_drop["LivingArea"] = dataset_total_last6_drop["LotSizeSquareFeet"].replace(0, np.nan)

In [28]:
# remove duplicate rows
data = dataset_total_last6_drop.drop_duplicates()

In [29]:
# replace some missing value with unknown so that we will not lose too moch important data 
cols = ["PropertySubType","City","CountyOrParish","StateOrProvince","FireplaceYN"]
dataset_total_last6_drop[cols] = dataset_total_last6_drop[cols].fillna("Unknown")

In [30]:
# the correlation between missing value rows
missing = dataset_total_last6_drop.isna().astype(int)
missing_corr = missing.corr()
missing_corr

,ClosePrice,Latitude,Longitude,PropertyType,LivingArea,CountyOrParish,ParkingTotal,PropertySubType,LotSizeAcres,YearBuilt,StreetNumberNumeric,BathroomsTotalInteger,City,BedroomsTotal,StateOrProvince,FireplaceYN,LotSizeArea,PostalCode,LotSizeSquareFeet,date_ym
ClosePrice,1.000000,-0.000130,-0.000127,NaN,0.000580,NaN,-0.001605,NaN,0.000503,0.001864,-0.000448,-0.002022,NaN,0.002155,NaN,NaN,-0.000056,-0.000148,0.000580,NaN
Latitude,-0.000130,1.000000,0.971820,NaN,0.012334,NaN,-0.002360,NaN,0.012107,0.013983,0.091114,-0.002971,NaN,0.008950,NaN,NaN,0.014600,-0.000218,0.012334,NaN
Longitude,-0.000127,0.971820,1.000000,NaN,0.012958,NaN,-0.002293,NaN,0.012727,0.010553,0.093793,-0.002888,NaN,0.009416,NaN,NaN,0.015259,-0.000212,0.012958,NaN
PropertyType,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LivingArea,0.000580,0.012334,0.012958,NaN,1.000000,NaN,-0.050024,NaN,0.986027,-0.038969,0.014703,-0.065573,NaN,-0.061764,NaN,NaN,0.887338,-0.002311,1.000000,NaN
CountyOrParish,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ParkingTotal,-0.001605,-0.002360,-0.002293,NaN,-0.050024,NaN,1.000000,NaN,-0.050959,0.669458,0.004067,0.793956,NaN,0.651697,NaN,NaN,-0.043148,-0.002687,-0.050024,NaN
PropertySubType,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LotSizeAcres,0.000503,0.012107,0.012727,NaN,0.986027,NaN,-0.050959,NaN,1.000000,-0.039440,0.015667,-0.066662,NaN,-0.062181,NaN,NaN,0.876272,-0.002408,0.986027,NaN
YearBuilt,0.001864,0.013983,0.010553,NaN,-0.038969,NaN,0.669458,NaN,-0.039440,1.000000,0.058890,0.527876,NaN,0.643709,NaN,NaN,-0.044096,0.004712,-0.038969,NaN


In [31]:
# drop nan values
dataset_total_last6_drop=dataset_total_last6_drop.dropna()
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 232951 entries, 1 to 23256
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ClosePrice             232951 non-null  float64       
 1   Latitude               232951 non-null  float64       
 2   Longitude              232951 non-null  float64       
 3   PropertyType           232951 non-null  object        
 4   LivingArea             232951 non-null  float64       
 5   CountyOrParish         232951 non-null  object        
 6   ParkingTotal           232951 non-null  float64       
 7   PropertySubType        232951 non-null  object        
 8   LotSizeAcres           232951 non-null  float64       
 9   YearBuilt              232951 non-null  float64       
 10  StreetNumberNumeric    232951 non-null  float64       
 11  BathroomsTotalInteger  232951 non-null  float64       
 12  City                   232951 non-null  object    

### 2. Convert datatype

In [32]:
dataset_total_last6_drop["FireplaceYN"].value_counts()

FireplaceYN
True       147113
False       82466
Unknown      3372
Name: count, dtype: int64

In [33]:
#dataset_total_last6_drop["ViewYN"].value_counts()

In [34]:
# yes or no
tf_map = {True: 1, False: 0, "Unknown":2}
#dataset_total_last6_drop["ViewYN"]=dataset_total_last6_drop["ViewYN"].map(tf_map)
dataset_total_last6_drop["FireplaceYN"]=dataset_total_last6_drop["FireplaceYN"].map(tf_map)

In [35]:
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 232951 entries, 1 to 23256
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ClosePrice             232951 non-null  float64       
 1   Latitude               232951 non-null  float64       
 2   Longitude              232951 non-null  float64       
 3   PropertyType           232951 non-null  object        
 4   LivingArea             232951 non-null  float64       
 5   CountyOrParish         232951 non-null  object        
 6   ParkingTotal           232951 non-null  float64       
 7   PropertySubType        232951 non-null  object        
 8   LotSizeAcres           232951 non-null  float64       
 9   YearBuilt              232951 non-null  float64       
 10  StreetNumberNumeric    232951 non-null  float64       
 11  BathroomsTotalInteger  232951 non-null  float64       
 12  City                   232951 non-null  object    

In [36]:
print(dataset_total_last6_drop["PropertyType"].value_counts())
dataset_total_last6_drop = pd.get_dummies( dataset_total_last6_drop, columns=["PropertyType"], drop_first=True)

PropertyType
Residential           171367
ResidentialLease       57944
ManufacturedInPark      2113
ResidentialIncome       1527
Name: count, dtype: int64


In [37]:
print(dataset_total_last6_drop["PropertySubType"].value_counts())
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
dataset_total_last6_drop["PropertySubType"] = le.fit_transform(dataset_total_last6_drop["PropertySubType"].astype(str))

PropertySubType
SingleFamilyResidence    166830
Condominium               34152
Townhouse                 12165
Apartment                  5788
Unknown                    4837
Duplex                     3335
ManufacturedOnLand         2593
Triplex                     989
Quadruplex                  766
StockCooperative            507
Cabin                       271
Studio                      184
MixedUse                    178
RoomingHouse                177
MobileHome                   45
OwnYourOwn                   37
Loft                         29
BoatSlip                     26
ManufacturedHome             24
CoOwnership                  10
Farm                          3
DeededParking                 3
Timeshare                     2
Name: count, dtype: int64


In [38]:
dataset_total_last6_drop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 232951 entries, 1 to 23256
Data columns (total 22 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   ClosePrice                      232951 non-null  float64       
 1   Latitude                        232951 non-null  float64       
 2   Longitude                       232951 non-null  float64       
 3   LivingArea                      232951 non-null  float64       
 4   CountyOrParish                  232951 non-null  object        
 5   ParkingTotal                    232951 non-null  float64       
 6   PropertySubType                 232951 non-null  int32         
 7   LotSizeAcres                    232951 non-null  float64       
 8   YearBuilt                       232951 non-null  float64       
 9   StreetNumberNumeric             232951 non-null  float64       
 10  BathroomsTotalInteger           232951 non-null  float64      

In [39]:
dataset_total_last6_drop.to_csv("cleaned_dataset.csv", index=False)

# 3. Test train split based on X monthes

— Training set: all months except the most recent complete month.

— Validation set: carve out the second-most-recent complete month from the training set for hyperparameter tuning and model selection — never tune on the test set.

— Test set: the last complete month of available data, touched only once, at the end.

In [40]:
X=2 # past 2 month
months = sorted(dataset_total_last6["date_ym"].unique())
data_train=dataset_total_last6[dataset_total_last6['date_ym'].isin(months[-X:-1])]
data_test=dataset_total_last6[dataset_total_last6['date_ym']==months[-1]]

In [41]:
print(data_train['date_ym'].value_counts())
print(data_test['date_ym'].value_counts())

date_ym
2026-04-01    23412
Name: count, dtype: int64
date_ym
2026-05-01    23260
Name: count, dtype: int64
